# 04 — Workflow Patterns

This notebook examines the three workflow types collected in Section J: independent job arrays, pipelines, and extended (long-running) calculations. It also covers checkpoint status for extended calculations.

**Run `00_setup.ipynb` first.**

In [ ]:
# Download shared analysis module from GitHub
!wget -q https://raw.githubusercontent.com/svaradh/hpc-questionnaire/main/analysis/sheets_client.py
SPREADSHEET_ID = 'PASTE_YOUR_SPREADSHEET_ID_HERE'  # ← change this

In [ ]:
from sheets_client import (
    load_sheets, summarise_sheets, explode_semicolons,
    split_semicolons, map_range_labels,
    CPU_HOURS_LABELS, WALL_TIME_LABELS, MEMORY_LABELS,
    MEMORY_PER_CORE_LABELS, CORES_LABELS, GPU_MEMORY_LABELS,
    JOB_COUNT_LABELS, CPU_HOURS_MIDPOINTS, JOB_COUNT_MIDPOINTS
)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

dfs = load_sheets(SPREADSHEET_ID)
print('Data loaded. Submissions:', len(dfs.get('Submissions', pd.DataFrame())))

---
## Chart 1 — Workflow Type Prevalence

Shows how many submission groups have entries in each workflow type tab. A group may appear in multiple workflow types (e.g. a group that runs both independent jobs and extended calculations).

**What to look for:** What proportion of groups run extended (long-wall-time) calculations? How many depend purely on throughput (independent jobs)?

In [ ]:
def count_unique_submissions(df: pd.DataFrame) -> int:
    for col in ['submission_id', 'submissionId']:
        if col in df.columns:
            return df[col].nunique()
    return len(df)  # fallback: count rows

type_counts = pd.Series({
    'Independent\njobs': count_unique_submissions(ind) if not ind.empty else 0,
    'Pipelines': count_unique_submissions(pip) if not pip.empty else 0,
    'Extended\ncalculations': count_unique_submissions(ext) if not ext.empty else 0,
})

fig, ax = plt.subplots(figsize=(7, 4))
colors = sns.color_palette('muted', n_colors=3)
bars = ax.bar(type_counts.index, type_counts.values, color=colors)
for bar, val in zip(bars, type_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2, str(val),
            ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('Number of groups with this workflow type')
ax.set_title('Workflow Type Prevalence\n(groups can appear in multiple types)', fontsize=12)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

---
## Chart 2 — Independent Job Wall Times

Distribution of wall times for independent job arrays. Only numeric entries are plotted.

**What to look for:** Are independent jobs short (< 24 h, compatible with standard queues) or longer? Long independent jobs may indicate the group does not decompose their workload as finely as possible, or may genuinely require longer per-job wall times.

In [ ]:
if ind.empty or 'wall_time_hours' not in ind.columns:
    print("No wall-time data in IndependentJobs.")
else:
    wt = pd.to_numeric(ind['wall_time_hours'], errors='coerce').dropna()
    wt = wt[wt > 0]

    if wt.empty:
        print("No numeric wall-time values found.")
    else:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(wt, bins=30, color=sns.color_palette('muted')[0], edgecolor='white')
        ax.axvline(24, color='tomato', linestyle='--', linewidth=1.2, label='24 h')
        ax.set_xlabel('Wall time per independent job (hours)')
        ax.set_ylabel('Number of job records')
        ax.set_title('Independent Job Wall-Time Distribution', fontsize=13)
        ax.legend()
        plt.tight_layout()
        plt.show()
        print(f"Records: {len(wt)}  |  Min: {wt.min():.1f} h  |  Median: {wt.median():.1f} h  |  Max: {wt.max():.1f} h")

---
## Chart 3 — CPU-Hours Used vs Needed (independent jobs)

Side-by-side bars comparing cpu_hours_used and cpu_hours_needed for independent job workflows. Range strings are mapped to midpoints for ordering.

**What to look for:** Is there a systematic gap between used and needed? A large gap indicates unmet computational demand that the current facility is not satisfying.

In [ ]:
CPU_HOURS_ORDER = list(CPU_HOURS_LABELS.values())

for col_pair in [('cpu_hours_used', 'cpu_hours_needed')]:
    used_col, needed_col = col_pair
    if ind.empty or used_col not in ind.columns or needed_col not in ind.columns:
        print(f"Columns {used_col} or {needed_col} not found in IndependentJobs.")
        break

    used_counts = (
        ind[used_col].dropna().str.strip().loc[lambda s: s != '']
    )
    needed_counts = (
        ind[needed_col].dropna().str.strip().loc[lambda s: s != '']
    )

    used_mapped = map_range_labels(used_counts, CPU_HOURS_LABELS).value_counts()
    needed_mapped = map_range_labels(needed_counts, CPU_HOURS_LABELS).value_counts()

    all_cats = [c for c in CPU_HOURS_ORDER if c in used_mapped.index or c in needed_mapped.index]

    df_plot = pd.DataFrame({
        'Used': used_mapped.reindex(all_cats, fill_value=0),
        'Needed': needed_mapped.reindex(all_cats, fill_value=0),
    }, index=all_cats)

    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(df_plot))
    width = 0.38
    b1 = ax.bar(x - width/2, df_plot['Used'], width, label='CPU-hours used', color=sns.color_palette('muted')[0])
    b2 = ax.bar(x + width/2, df_plot['Needed'], width, label='CPU-hours needed', color=sns.color_palette('muted')[1])
    for bar in list(b1) + list(b2):
        if bar.get_height() > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, str(int(bar.get_height())),
                    ha='center', va='bottom', fontsize=7)
    ax.set_xticks(x)
    ax.set_xticklabels(df_plot.index, rotation=15, ha='right')
    ax.set_xlabel('CPU-hours per year')
    ax.set_ylabel('Number of entries')
    ax.set_title('CPU-Hours Used vs Needed — Independent Jobs', fontsize=13)
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## Chart 4 — Extended Calculation Wall Times vs Longest Available

Side-by-side comparison of the actual wall time needed for extended calculations vs the longest queue limit available to the group. This directly shows the wall-time gap.

**What to look for:** Where wall_time_hours exceeds longest_available_hours, the group faces a genuine wall-time constraint. These are primary candidates for a long-wall-time QoS class.

In [ ]:
if ext.empty:
    print("No ExtendedCalcs data available.")
elif 'wall_time_hours' not in ext.columns or 'longest_available_hours' not in ext.columns:
    print("wall_time_hours or longest_available_hours not found in ExtendedCalcs.")
else:
    df_ext = ext[['wall_time_hours', 'longest_available_hours']].copy()
    df_ext['wall_time_hours'] = pd.to_numeric(df_ext['wall_time_hours'], errors='coerce')
    df_ext['longest_available_hours'] = pd.to_numeric(df_ext['longest_available_hours'], errors='coerce')
    df_ext = df_ext.dropna(subset=['wall_time_hours'])
    df_ext = df_ext[df_ext['wall_time_hours'] > 0]
    df_ext = df_ext.reset_index(drop=True)

    if df_ext.empty:
        print("No valid extended calculation wall-time records.")
    else:
        fig, ax = plt.subplots(figsize=(10, 4))
        x = np.arange(len(df_ext))
        width = 0.38
        ax.bar(x - width/2, df_ext['wall_time_hours'], width, label='Wall time needed (h)', color='steelblue')
        ax.bar(
            x + width/2,
            df_ext['longest_available_hours'].fillna(0),
            width,
            label='Longest available (h)',
            color='lightcoral',
        )
        ax.set_xlabel('Extended calculation entry index')
        ax.set_ylabel('Wall time (hours)')
        ax.set_title('Extended Calculations: Wall Time Needed vs Longest Available', fontsize=12)
        ax.legend()
        plt.tight_layout()
        plt.show()

        gap = (df_ext['wall_time_hours'] > df_ext['longest_available_hours'].fillna(0))
        print(f"Entries where needed > available: {gap.sum()} of {len(df_ext)}")

---
## Chart 5 — Checkpoint Status for Extended Calculations

Distribution of checkpoint capability across extended calculation entries.

**What to look for:** For workloads that exceed available wall-time limits, can they checkpoint? If not, a long-wall-time QoS class is technically necessary — checkpoint/restart is not a substitute.

In [ ]:
CHECKPOINT_LABELS = {
    'yes': 'Yes — checkpointing available',
    'no': 'No — cannot checkpoint',
    'dont_know': "Don't know",
    'not_tested': 'Not tested',
    'partial': 'Partial / limited',
}

if ext.empty or 'can_checkpoint' not in ext.columns:
    print("No checkpoint status data in ExtendedCalcs.")
else:
    cp = (
        ext['can_checkpoint']
        .dropna().str.strip()
        .loc[lambda s: s != '']
        .map(lambda x: CHECKPOINT_LABELS.get(x, x))
        .value_counts()
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(cp.index, cp.values, color=sns.color_palette('Set2', n_colors=len(cp)))
    for bar, val in zip(bars, cp.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05, str(val),
                ha='center', va='bottom', fontsize=9)
    ax.set_xlabel('Checkpoint capability')
    ax.set_ylabel('Number of extended calculation entries')
    ax.set_title('Checkpoint Availability for Extended Calculations', fontsize=12)
    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    plt.show()

---
## Chart 6 — Pipeline Stage Count Distribution

Distribution of the number of stages in pipeline workflows. Numeric entries only.

**What to look for:** Multi-stage pipelines where intermediate stages have long wall times may generate complex scheduling requirements. Many-stage pipelines may benefit from workflow management tools.

In [ ]:
if pip.empty or 'stages' not in pip.columns:
    print("No pipeline stage data available.")
else:
    stages = pd.to_numeric(pip['stages'], errors='coerce').dropna()
    stages = stages[stages >= 1]

    if stages.empty:
        print("No numeric stage count values found.")
    else:
        stage_counts = stages.value_counts().sort_index()

        fig, ax = plt.subplots(figsize=(8, 4))
        bars = ax.bar(stage_counts.index.astype(int), stage_counts.values,
                      color=sns.color_palette('muted')[3])
        for bar, val in zip(bars, stage_counts.values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05, str(val),
                    ha='center', va='bottom', fontsize=9)
        ax.set_xlabel('Number of pipeline stages')
        ax.set_ylabel('Number of pipeline entries')
        ax.set_title('Pipeline Stage Count Distribution', fontsize=13)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        plt.tight_layout()
        plt.show()
        print(f"Records: {len(stages)}  |  Min: {stages.min():.0f}  |  Median: {stages.median():.0f}  |  Max: {stages.max():.0f}")